# 📘 Домашнє завдання №23. Інструменти для роботи з LLM: LangChain та RAG

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW24

## Завдання

Створити просту RAG-систему (Retrieval-Augmented Generation), яка відповідає на питання за текстом Конституції України:

https://unece.org/fileadmin/DAM/hlm/prgm/cph/experts/ukraine/ukr.constitution.e.pdf


### Що потрібно реалізувати

Вам необхідно побудувати систему, яка:

1. Завантажує PDF документ
2. Розбиває його на фрагменти (chunking)
3. Перетворює текст у embeddings
4. Зберігає їх у векторній базі даних (FAISS або Chroma)
5. Виконує semantic search по запиту користувача
6. Передає знайдений контекст у LLM
7. Генерує відповідь на основі Конституції України

### Рекомендований стек

Обробка документа:
- `PyPDFLoader` або аналог

Chunking:
- `RecursiveCharacterTextSplitter`

Embeddings:
- `sentence-transformers/all-MiniLM-L6-v2`
- або multilingual варіант:
  - `intfloat/multilingual-e5-base`


### Рекомендовані LLM (для генерації відповідей)

Простий варіант:
- `Qwen/Qwen2.5-1.5B-Instruct`

Кращі варіанти для україномовного RAG:
- `Qwen/Qwen2.5-7B-Instruct`

### Вимоги до реалізації

- Використати vector database (`FAISS` або `Chroma`)
- Реалізувати retrieval (`top-k` = 3–5 документів)
- Передати контекст у prompt
- Забезпечити відповідь виключно на основі документа

### Тестові питання

Перевірте вашу систему на таких запитах:

1. Які основні права і свободи людини гарантує Конституція України?

2. Яка структура державної влади описана в Конституції України?

3. Які умови зміни Конституції України передбачені в документі?

### Порада

Якщо плануєте україномовний RAG — використовуйте мультимовні embeddings та сучасні instruct-моделі, оскільки якість відповіді залежить не тільки від retrieval, але й від мовної моделі.

In [1]:
# # Silent installation or update
#
# # Clean cache
# !python3 -m pip cache purge -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
#     "kagglehub",
#     "kagglesdk",
# ]
#
# for package_name in package_update:
#     !bash -c "python3 -m pip install -U '{package_name}' -q"
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "kagglehub[pandas-datasets]",
#     "numpy",
#     "matplotlib",
#     "scipy",
#     "statsmodels",
#     "joblib",
#     "torch",
#     "transformers",
#     "pypdf",
#     "unstructured[pdf]",
#     "langchain-unstructured",
#     "langchain_text_splitters",
#     "langchain_huggingface",
#     "sentence_transformers",
#     "chromadb",
#     "langchain-chroma",
# ]
#
# for package_name in package_array:
#     !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [2]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input data
hm_version = 24
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone data
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"

# ✅ Крок: 1. Завантажити PDF документ

In [3]:
# Load document

import requests
import apps.reporter as rpt
from pathlib import Path
from langchain_unstructured.document_loaders import UnstructuredLoader

# Input data
df_file_name = "Файл Конституції України"
load_file = True
file_name = "resources/ukr.constitution.ua.pdf"
file_url = "https://dduvs.edu.ua/wp-content/uploads/files/Structure/f/fpfpmgb/p/d1.pdf"

# Solution
file_path = Path(file_name)

# try to load file
if load_file and file_path.exists():
    loader = UnstructuredLoader(file_path, strategy="fast", languages=["ukr"])
else:
    response = requests.get(file_url, stream=True)
    # save file
    if response.status_code == 200:
        with open(file_path, "wb") as current_file:
            current_file.write(response.content)
    loader = UnstructuredLoader(file_path, strategy="fast", languages=["ukr"])

docs = loader.load()

rp = rpt.Reporter()
rp.add_item("Розмір документа", str(len(docs)))

# Print results
rp.print_pd_report(df_file_name)

INFO: pikepdf C++ to Python logger bridge initialized


Attribute,Result
Розмір документа,1386


# ✅ Крок: 2. Розбити його на фрагменти (chunking)

In [4]:
# Splitting data

import apps.reporter as rpt
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Input data

# Solution
# обʼєднуємо дані попередньо розбиті
full_text = "\n\n".join([doc.page_content for doc in docs])

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

# розбиваємо на чанки
chunks = splitter.split_text(full_text)

rp = rpt.Reporter()
rp.add_item("Кількість чанків", str(len(chunks)))

# Print results
rp.print_pd_report(df_file_name)


Attribute,Result
Кількість чанків,169


# ✅ Крок: 3. Перетворити текст в embeddings

In [5]:
# Embeddings

import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

# Input data
load_model = True
model_file_name = "resources/embedding_model"

# Solution
file_path = Path(model_file_name)

# try to load model
if load_model and file_path.exists():
    embeddings = HuggingFaceEmbeddings(
        model_name=str(file_path),
        encode_kwargs={'prompt_name': 'document'},
        model_kwargs={"local_files_only": True},
    )
else:
    model = SentenceTransformer("intfloat/multilingual-e5-base")
    # save model
    model.save(str(file_path))

    embeddings = HuggingFaceEmbeddings(
        model_name=str(file_path),
        encode_kwargs={'prompt_name': 'document'},
        model_kwargs={"local_files_only": True},
    )

# Print results


INFO: No device provided, using mps
INFO: Loading SentenceTransformer model from resources/embedding_model.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# ✅ Крок: 4. Зберегти їх у векторній базі даних (FAISS або Chroma)

In [6]:
# Vectorization & Retriever

import torch
from langchain_chroma import Chroma

# Input data
n_docs = 5

# Solution
db_vectors = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings
)
retriever = db_vectors.as_retriever(
    search_type="similarity",
    search_kwargs={"k": n_docs}
)

# Print results


# ✅ Крок: 5. Виконати semantic search по запиту користувача

In [7]:
# Semantic Search test

# Input data
n_docs = 5
query = "Які права має людина?"

# Solution
found_results = db_vectors.similarity_search(query, k=n_docs)

rp = rpt.Reporter("Промпт для моделі", "Відповідь")

for result in found_results:
    rp.add_item(query, result.page_content)

# Print results
# rp.print_pd_report(f"{n_docs} результатів пошуку фрагментів")


# ✅ Крок: 6. Передає знайдений контекст у LLM

In [8]:
# Context

import torch
import apps.reporter as rpt
from pathlib import Path
from transformers import pipeline, GenerationConfig

# Input data
load_model = True
model_file_name = "resources/instruction_understanding_model"

query = "Які права має людина?"

# Solution
file_path = Path(model_file_name)

# try to load model
if load_model and file_path.exists():
    model = pipeline(
        "text-generation",
        model=model_file_name,
        tokenizer=model_file_name,
        clean_up_tokenization_spaces=False,
    )
else:
    model = pipeline(
        "text-generation",
        # model="Qwen/Qwen2.5-1.5B-Instruct",
        # model="Qwen/Qwen2.5-7B-Instruct",
        model="Radu1999/Mistral-Instruct-Ukrainian-SFT",
        clean_up_tokenization_spaces=False,
    )
    # save model
    model.model.save_pretrained(model_file_name)
    model.tokenizer.save_pretrained(model_file_name)


def build_prompt(context, question):
    return f"""
    Ви є помічником-юристом.
    Дай коротку відповідь на питання, використовуючи наданий контекст.

    Контекст:
    {context}

    Запитання:
    {question}

    Відповідь:
    """


def rag_answer(question):
    found_results = retriever.invoke(question)
    context_text = "\n\n".join([doc.page_content for doc in found_results])

    prompt = build_prompt(context_text, question)

    config = GenerationConfig(
        max_new_tokens=256,
        temperature=None,
        top_p=None,
        top_k=None,
        do_sample=False,
        no_repeat_ngram_size=3,
        pad_token_id=model.tokenizer.eos_token_id,
    )

    return model(prompt, generation_config=config)[0]["generated_text"][len(prompt):].strip()


rp = rpt.Reporter("Питання для моделі", "Відповідь")
rp.add_item(query, rag_answer(query))

# Print results
rp.print_pd_report(f"Перевірка роботи моделі")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Питання для моделі,Відповідь
Які права має людина?,"Кожну людину мають визнати вільною і рідною в свої гідности і права. Її прав не можна зменшувати або обмежувати. Права людині - це право на розвивальний вільності, право на безпеку, право на живу і збагачуватися, право бути здоровим, право мати сім’ю, право виховувати дітей, право вільного вираження думки, право свободно обирати і бути обиравним, право на справедливий суд, право займатися релігією, право здобувати освіту, право на мирне життя, право не бути позбуваним життя без суду, право працювати і заробляти, право володіти майном, право переміщатися, права на національну культуру, право обиватися в своєму національному одязі"


# ✅ Крок: 7. Генерує відповідь на основі Конституції України

In [ ]:
# Questions

import apps.reporter as rpt

# Input data
questions = [
    "Які основні права і свободи людини гарантує Конституція України?",
    "Яка структура державної влади описана в Конституції України?",
    "Які умови зміни Конституції України передбачені в документі?",
]

# Solution
rp = rpt.Reporter("Питання для моделі", "Відповідь")

for question in questions:
    rp.add_item(question, rag_answer(question))

# Print results
rp.print_pd_report(f"Відповіді на питання")


Висновок. Потужніші моделі вимагають більше ресурсів, дана модель "Qwen/Qwen2.5-1.5B-Instruct", яка не призведе до збою через обмеження ресурсів, дуже схильна до суржику, мовних помилок, вигадування слів, втрачає звʼязок між словами. З переваг є те, що модель пробувала використовувати дані з контексту. Загалом пошукова система працює правильно, але перепоною в наданні правильних відповідей є ресурсні обмеження, які не дозволяють використати моделі більшої потужності. Набагато кращі результати показала модель "Radu1999/Mistral-Instruct-Ukrainian-SFT", але на формування відповіді вона витрачає значно більше часу.